# Lesson 03 - Modèles de Conception Agentique

Dans cette leçon, nous explorons trois modèles de conception fondamentaux pour construire des agents IA efficaces :

1. **Instructions Claires pour l'Agent** — Élaboration de prompts précis définissant des rôles qui guident le comportement de l'agent  
2. **Sortie Structurée avec des Modèles Pydantic** — Assurer que les agents retournent des données prévisibles et validées  
3. **Agents à Responsabilité Unique** — Concevoir des agents ciblés qui font chacun bien une seule chose  

Nous appliquerons chaque modèle à un scénario de **recommandation de destinations de voyage**, en construisant progressivement un système capable de suggérer des destinations, vérifier leur disponibilité, et gérer la logistique.


## Configuration


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Modèle 1 : Instructions claires pour l’agent

Le modèle le plus efficace est aussi le plus simple : écrire des instructions claires et détaillées pour votre agent.

De bonnes instructions définissent :
- **Qui** est l’agent (persona et ton)
- **Ce qu’**il doit faire (responsabilités étape par étape)
- **Comment** il doit se comporter (contraintes et style)

Ci-dessous, nous créons un agent concierge de voyage avec des instructions explicites qui façonnent chaque réponse qu’il produit.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Modèle 2 : Sortie Structurée avec les Modèles Pydantic

Le texte libre est utile pour la conversation, mais les systèmes en aval ont besoin de données structurées.  
En associant **les modèles Pydantic** avec une **fonction outil**, nous pouvons :

- Définir un schéma exact pour la sortie de l'agent  
- Valider automatiquement les réponses  
- Intégrer de manière fiable les résultats de l'agent dans la logique applicative  

Nous présentons également un outil qui renvoie les détails de la destination afin que l'agent base ses recommandations sur des données réelles.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Modèle 3 : Agents à responsabilité unique

Les tâches complexes bénéficient de la répartition du travail entre plusieurs agents spécialisés, chacun ayant une responsabilité unique :

- Un **Expert en destination** qui connaît les lieux et leur disponibilité
- Un **Planificateur logistique** qui gère les vols, les hôtels et les itinéraires

Cela reflète le principe d'ingénierie logicielle de *séparation des préoccupations* — chaque agent est plus facile à tester, maintenir et améliorer indépendamment.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Résumé

Dans cette leçon, nous avons appliqué trois modèles de conception agentique à un scénario de recommandation de voyage :

| Modèle | Idée clé | Bénéfice |
|---|---|---|
| **Instructions Claires** | Définir la persona, les responsabilités et les contraintes dès le départ | Comportement d’agent cohérent et conforme à la marque |
| **Sortie Structurée** | Utiliser des modèles Pydantic comme format de réponse | Résultats validés et lisibles par machine |
| **Responsabilité Unique** | Donner à chaque agent une tâche ciblée | Plus facile à tester, maintenir et composer |

Ces modèles se combinent naturellement — vous pouvez associer des instructions claires avec une sortie structurée dans un agent à responsabilité unique pour construire des systèmes robustes prêts pour la production.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :  
Ce document a été traduit à l’aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforcions d’assurer l’exactitude, veuillez noter que les traductions automatiques peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue d’origine doit être considéré comme la source faisant autorité. Pour les informations critiques, une traduction professionnelle humaine est recommandée. Nous déclinons toute responsabilité en cas de malentendus ou de mauvaises interprétations résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
